In [ ]:
%%sql -r dataframe_1
--> Run these first
USE DATABASE RESEARCH_COMPASS;
USE SCHEMA RAG;
USE WAREHOUSE COMPUTE_WH;

# Research Compass

## Phase 1 — Dynamic Ingestion Pipeline

In [ ]:
from snowflake.snowpark.context import get_active_session
import hashlib

session = get_active_session()

def ingest_paper(filename, title=None):
    # Step 1: Generate a unique paper_id from filename
    paper_id = hashlib.md5(filename.encode()).hexdigest()[:8]
    
    # Step 2: Check if already ingested
    existing = session.sql(f"""
        SELECT COUNT(*) AS cnt 
        FROM RESEARCH_COMPASS.RAG.PAPERS 
        WHERE paper_id = '{paper_id}'
    """).collect()
    
    if existing[0]["CNT"] > 0:
        print(f"Paper '{filename}' already ingested. Skipping.")
        return paper_id
    
    # Step 3: Parse the PDF
    print(f"Parsing '{filename}'...")
    parsed = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
            @RESEARCH_COMPASS.RAG.PDF_STAGE,
            '{filename}',
            {{'mode': 'LAYOUT'}}
        ) AS parsed_content
    """).collect()
    
    content = parsed[0]["PARSED_CONTENT"]
    
    # Step 4: Chunk the text
    print(f"Chunking...")
    session.sql(f"""
        INSERT INTO RESEARCH_COMPASS.RAG.CHUNKED_PAPERS
            (paper_id, filename, chunk_index, chunk_text)
        SELECT
            '{paper_id}',
            '{filename}',
            chunk.index,
            chunk.value::STRING
        FROM LATERAL FLATTEN(
            INPUT => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
                PARSE_JSON('{content}'):content::STRING,
                'markdown',
                500,
                50
            )
        ) AS chunk
    """).collect()
    
    # Step 5: Register in papers metadata table
    display_title = title if title else filename.replace('.pdf', '').replace('_', ' ')
    session.sql(f"""
        INSERT INTO RESEARCH_COMPASS.RAG.PAPERS 
            (paper_id, filename, title)
        VALUES 
            ('{paper_id}', '{filename}', '{display_title}')
    """).collect()
    
    # Step 6: Confirm
    chunk_count = session.sql(f"""
        SELECT COUNT(*) AS cnt 
        FROM RESEARCH_COMPASS.RAG.CHUNKED_PAPERS 
        WHERE paper_id = '{paper_id}'
    """).collect()[0]["CNT"]
    
    print(f"Done. '{display_title}' ingested with {chunk_count} chunks. Paper ID: {paper_id}")
    return paper_id

print("Ingestion pipeline ready")

## Phase 2 — RAG Pipeline

In [ ]:
import json

def retrieve_chunks(query, paper_id=None, limit=5):
    filter_clause = ""
    if paper_id:
        filter_clause = f', "filter": {{"@eq": {{"PAPER_ID": "{paper_id}"}}}}'
    
    result = session.sql(f"""
        SELECT PARSE_JSON(
            SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
                'RESEARCH_COMPASS.RAG.PAPER_SEARCH_SERVICE',
                '{{"query": "{query}", "columns": ["chunk_text", "paper_id", "filename"], "limit": {limit}{filter_clause}}}'
            )
        )['results'] AS results
    """).collect()
    
    raw = result[0]["RESULTS"]
    if raw is None:
        return []
    return json.loads(raw)

def retrieve_chunks_hybrid(query, paper_id=None, limit=5):
    # Semantic retrieval
    semantic_chunks = retrieve_chunks(query, paper_id, limit)
    
    # Get filenames from semantic results to avoid duplicates
    semantic_texts = set([c["chunk_text"][:50] for c in semantic_chunks])
    
    paper_clause = f"AND paper_id = '{paper_id}'" if paper_id else ""
    
    # Split the query and keep only words longer that 4 characters
    keywords = [w for w in query.split() if len(w) > 4]
    
    # Create SQL condition for Keyword search
    keyword_conditions = " OR ".join([
        f"LOWER(chunk_text) LIKE LOWER('%{kw}%')" for kw in keywords
    ])

    # Run SQL query on Chunks table
    if keyword_conditions:
        keyword_rows = session.sql(f"""
            SELECT chunk_text, paper_id, filename, chunk_index
            FROM RESEARCH_COMPASS.RAG.CHUNKED_PAPERS
            WHERE ({keyword_conditions})
            {paper_clause}
            LIMIT {limit}
        """).collect()
        
        # Deduplicate against semantic results using text prefix
        keyword_chunks = [
            {"chunk_text": r["CHUNK_TEXT"],
             "paper_id": r["PAPER_ID"],
             "filename": r["FILENAME"],
             "chunk_index": r["CHUNK_INDEX"]}
            for r in keyword_rows
            if r["CHUNK_TEXT"][:50] not in semantic_texts
        ]
    else:
        keyword_chunks = []
    
    return semantic_chunks + keyword_chunks


def retrieve_chunks_hybrid_expanded(query, paper_id=None, limit=5, max_chunks=20):
    semantic_chunks = retrieve_chunks(query, paper_id, limit)
    if not semantic_chunks:
        return []
    
    paper_clause = f"AND paper_id = '{paper_id}'" if paper_id else ""
    semantic_texts_escaped = [c["chunk_text"][:100].replace("'", "''") for c in semantic_chunks]
    text_conditions = " OR ".join([f"LEFT(chunk_text, 100) = '{t}'" for t in semantic_texts_escaped])
    
    index_rows = session.sql(f"""
        SELECT chunk_text, paper_id, filename, chunk_index
        FROM RESEARCH_COMPASS.RAG.CHUNKED_PAPERS
        WHERE ({text_conditions})
        {paper_clause}
    """).collect()
    
    semantic_with_index = [{"chunk_text": r["CHUNK_TEXT"],
                             "paper_id": r["PAPER_ID"],
                             "filename": r["FILENAME"],
                             "chunk_index": r["CHUNK_INDEX"]} for r in index_rows]
    
    semantic_indices = set([int(c["chunk_index"]) for c in semantic_with_index])
    expanded_indices = set(semantic_indices)
    
    header_markers = [
        "abstract", "introduction", "conclusion", "related work",
        "methodology", "results", "algorithm", "our approach",
        "proposed method", "debiasing", "experiments", "discussion"
    ]
    
    for c in semantic_with_index:
        idx = int(c["chunk_index"])
        text = c["chunk_text"].lower()
        is_header = any(marker in text for marker in header_markers)
        
        # Guaranteed minimum: always expand 4 forward
        # Headers: expand 8 forward to capture full sections
        forward = 8 if is_header else 4
        for offset in range(1, forward + 1):
            expanded_indices.add(idx + offset)
    
    indices_str = ','.join([str(i) for i in sorted(expanded_indices)])
    
    expanded_rows = session.sql(f"""
        SELECT chunk_text, paper_id, filename, chunk_index
        FROM RESEARCH_COMPASS.RAG.CHUNKED_PAPERS
        WHERE chunk_index IN ({indices_str})
        {paper_clause}
        ORDER BY chunk_index
        LIMIT {max_chunks}
    """).collect()
    
    expanded_chunks = [{"chunk_text": r["CHUNK_TEXT"],
                        "paper_id": r["PAPER_ID"],
                        "filename": r["FILENAME"],
                        "chunk_index": r["CHUNK_INDEX"]} for r in expanded_rows]
    
    # Add keyword chunks not already included
    hybrid_chunks = retrieve_chunks_hybrid(query, paper_id, limit)
    existing_keys = set([(c.get("filename"), c.get("chunk_index")) for c in expanded_chunks])
    for c in hybrid_chunks:
        if (c.get("filename"), c.get("chunk_index")) not in existing_keys:
            expanded_chunks.append(c)
    
    # Sort by filename then chunk_index
    expanded_chunks = sorted(
        expanded_chunks,
        key=lambda x: (x.get("filename", ""), int(x.get("chunk_index", 0)))
    )
    
    return expanded_chunks[:max_chunks]

print("Retrieval functions ready")

In [ ]:
def rag_query(query, paper_id=None):
    chunks = retrieve_chunks_hybrid_expanded(query, paper_id, limit=5, max_chunks=20)
    
    # Sort by filename then chunk_index to keep same-paper chunks together
    chunks = sorted(
        chunks,
        key=lambda x: (x.get("filename", ""), int(x.get("chunk_index", 0)))
    )
    
    if not chunks:
        return {
            "answer": "No papers have been uploaded yet. Please upload a paper first.",
            "sources": [],
            "chunks_used": 0
        }
    
    prompt = build_prompt(query, chunks)
    
    response = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            '{prompt.replace("'", "''")}'
        ) AS answer
    """).collect()
    
    sources = list(set([c.get("filename", "Unknown") for c in chunks]))
    
    return {
        "answer": response[0]["ANSWER"],
        "sources": sources,
        "chunks_used": len(chunks)
    }

print("RAG query ready")

In [ ]:
def build_prompt(query, chunks):
    context = ""
    for i, chunk in enumerate(chunks):
        filename = chunk.get("filename", "Unknown")
        text = chunk.get("chunk_text", "")
        context += f"[Chunk {i+1} - {filename}]\n{text}\n\n"
    
    return f"""You are an academic research assistant. Answer the user's question based ONLY on the provided context from the uploaded research papers.

If the answer is not in the context, say "I could not find information about this in the uploaded papers."

Always cite which paper your answer comes from.

Context:
{context}

Question: {query}

Answer:"""

def rag_query(query, paper_id=None):
    chunks = retrieve_chunks_hybrid_expanded(query, paper_id, limit=5, max_chunks=20)
    
    if not chunks:
        return {
            "answer": "No papers have been uploaded yet. Please upload a paper first.",
            "sources": [],
            "chunks_used": 0
        }
    
    prompt = build_prompt(query, chunks)
    
    response = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            '{prompt.replace("'", "''")}'
        ) AS answer
    """).collect()
    
    sources = list(set([c.get("filename", "Unknown") for c in chunks]))
    
    return {
        "answer": response[0]["ANSWER"],
        "sources": sources,
        "chunks_used": len(chunks)
    }

print("RAG pipeline ready")

### 2.1 — HyDE (Hypothetical Document Embedding)

In [ ]:
def generate_hypothetical_answer(query):
    prompt = f"""You are an academic research assistant. 
A user is looking for information in research papers about the following question:

"{query}"

Write a short hypothetical answer (3-5 sentences) that such a paper might contain.
Use academic language and terminology that would appear in a research paper.
Do not say "I think" or "perhaps" — write it as if it is a factual excerpt from a paper.

Hypothetical answer:"""

    response = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            '{prompt.replace("'", "''")}'
        ) AS hypothetical_answer
    """).collect()
    
    return response[0]["HYPOTHETICAL_ANSWER"].strip()

print("HyDE function ready")

In [ ]:
def rag_query_hyde(query, paper_id=None):
    hypothetical = generate_hypothetical_answer(query)
    
    hypothetical_clean = (hypothetical
        .replace('"', ' ')
        .replace("'", ' ')
        .replace('\n', ' ')
        .replace('\r', ' ')
        .strip()
    )
    
    hyde_chunks = retrieve_chunks_hybrid_expanded(
        hypothetical_clean, paper_id, limit=5, max_chunks=10
    )
    original_chunks = retrieve_chunks_hybrid_expanded(
        query, paper_id, limit=5, max_chunks=10
    )
    
    # Deduplicate
    seen = set()
    all_chunks = []
    for c in hyde_chunks + original_chunks:
        key = (c.get("filename"), c.get("chunk_index"))
        if key not in seen:
            seen.add(key)
            all_chunks.append(c)
    
    # Sort by filename first, then chunk_index
    # This keeps chunks from the same paper together
    all_chunks = sorted(
        all_chunks,
        key=lambda x: (x.get("filename", ""), int(x.get("chunk_index", 0)))
    )[:20]
    
    if not all_chunks:
        return {
            "answer": "No relevant information found in the uploaded papers.",
            "sources": [],
            "chunks_used": 0
        }
    
    prompt = build_prompt(query, all_chunks)
    
    response = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            '{prompt.replace("'", "''")}'
        ) AS answer
    """).collect()
    
    sources = list(set([c.get("filename", "Unknown") for c in all_chunks]))
    
    return {
        "answer": response[0]["ANSWER"],
        "sources": sources,
        "chunks_used": len(all_chunks),
        "hypothetical": hypothetical
    }

print("HyDE pipeline ready")

In [ ]:
result = rag_query_hyde(
    "What are the trade-offs between hard debiasing and soft debiasing?"
)
print(f"Chunks used: {result['chunks_used']}")
print(f"\nAnswer:\n{result['answer']}")